# Evaluación Parcial 1 - Machine Learning

**Asignatura:** MLY0100 Machine Learning  
**Integrantes:** Por completar  
**Dataset:** DS1-18-Datos-Properati.csv  
**Fecha:** Por completar


# Fase 1 - Comprensión del Negocio

## Contexto

El dataset utilizado contiene publicaciones de propiedades ubicadas en Argentina. Incluye información relacionada con la ubicación de los inmuebles, cantidad de ambientes, dormitorios y baños, superficies, precio, tipo de propiedad y tipo de operación, entre otras variables.

En esta primera etapa del proyecto se busca comprender estos datos antes de construir modelos de Machine Learning. Para esto se realizará un análisis exploratorio que permita conocer cómo están distribuidas las propiedades y sus precios, detectar datos faltantes y valores atípicos, y preparar posteriormente la información para que pueda ser utilizada en tareas de regresión y clasificación.

## Objetivo

Analizar las principales características de las propiedades del dataset para comprender su comportamiento, especialmente en relación con el precio, la ubicación, las superficies y el tipo de propiedad. También se busca detectar problemas de calidad de los datos, como valores faltantes y valores atípicos, para dejar la información preparada para etapas posteriores de Machine Learning.

## Pregunta analítica

¿Qué características de las propiedades, como la ubicación, la superficie y el tipo de propiedad, se relacionan con las diferencias observadas en sus precios de venta?

## Supuestos

Para comenzar el análisis se consideran los siguientes supuestos de trabajo, los cuales se revisarán durante la comprensión de los datos:

- Los registros corresponden a publicaciones de propiedades en venta en Argentina.
- El precio informado representa el valor publicado de cada propiedad y será analizado junto con variables como ubicación, superficie y tipo de propiedad.
- Los valores faltantes no se interpretarán como cero, sino que se estudiarán antes de decidir su tratamiento.
- Los valores muy altos o muy bajos no se eliminarán automáticamente, ya que primero se debe verificar si corresponden a errores o a propiedades reales con características diferentes.
- Las conclusiones de esta etapa se limitarán al dataset entregado y no se tomarán como una representación completa de todo el mercado inmobiliario argentino.

## Variable objetivo para regresión

Para una futura tarea de regresión se propone utilizar **price** como variable objetivo.

Esta variable es adecuada porque contiene valores numéricos continuos que representan el precio publicado de cada propiedad. El objetivo de un modelo de regresión sería estimar ese valor a partir de otras características del inmueble, por ejemplo su ubicación, superficie, cantidad de ambientes, dormitorios, baños y tipo de propiedad.

En esta evaluación no se entrenará todavía el modelo. En esta etapa solamente se identifica y justifica la variable objetivo, mientras se analiza y prepara el dataset.

## Variable objetivo para clasificación

Para una futura tarea de clasificación se propone utilizar **property_type** como variable objetivo.

Esta variable es adecuada porque representa categorías discretas de propiedades. Un modelo de clasificación podría intentar identificar el tipo de propiedad a partir de características como la superficie, cantidad de ambientes, dormitorios, baños, ubicación y precio.

Al igual que en el caso de regresión, en esta evaluación no se entrenará todavía el modelo. En esta etapa solamente se identifica y justifica el target de clasificación.


# Fase 2 - Comprensión de los Datos

## Carga de librerías y datos

Antes de analizar el dataset se importan las librerías que se utilizarán durante el trabajo. Luego se carga el archivo CSV que está dentro del ZIP entregado por el profesor. Se mantiene el archivo original sin modificaciones en esta etapa, porque primero necesitamos conocer su estructura y revisar la calidad de los datos.


In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Buscar el ZIP en ubicaciones comunes del proyecto o Google Colab
candidatos = []
for carpeta in [Path('.'), Path('..'), Path('/content')]:
    candidatos.extend(carpeta.glob('Evaluación Parcial 1 - Dataset*.zip'))

if not candidatos:
    raise FileNotFoundError('No se encontró el ZIP del dataset.')

ruta_zip = candidatos[0]

with zipfile.ZipFile(ruta_zip, 'r') as archivo_zip:
    nombre_csv = next(
        nombre for nombre in archivo_zip.namelist()
        if nombre.endswith('DS1-18-Datos-Properati.csv')
        and not nombre.startswith('__MACOSX')
    )
    with archivo_zip.open(nombre_csv) as archivo_csv:
        df = pd.read_csv(archivo_csv)

print('Dimensiones del dataset:', df.shape)
df.head()


### Resultado de la carga

El dataset contiene **146.660 registros y 19 columnas**. La carga se realizó correctamente y las primeras filas muestran variables relacionadas con fechas, ubicación, ambientes, superficies, precio y tipo de propiedad. En este punto todavía no se realizan cambios sobre los datos; primero se revisará su estructura en detalle.


## Inspección de la estructura

En este bloque se revisan los nombres de las columnas, tipos de datos, cantidad de valores no nulos y un resumen general del dataset. Esta revisión permite detectar desde el inicio variables que necesitan correcciones de tipo antes de comenzar la limpieza y el análisis estadístico.


In [ ]:
# Revisar columnas y tipos de datos
print('Columnas del dataset:')
print(df.columns.tolist())

print('\nTipos de datos:')
print(df.dtypes)

print('\nInformación general:')
df.info()

# Resumen de variables numéricas y categóricas
columnas_numericas = df.select_dtypes(include='number').columns.tolist()
columnas_texto = df.select_dtypes(include='object').columns.tolist()

print('\nVariables numéricas:', columnas_numericas)
print('Variables almacenadas como texto/object:', columnas_texto)

df.describe(include='all').T


### Hallazgos de la estructura

El dataset tiene **19 columnas**. Actualmente se observan **8 variables de tipo `float64` y 11 de tipo `object`**.

Las variables `start_date`, `end_date` y `created_on` están almacenadas como texto, aunque representan fechas. Más adelante será necesario convertirlas a `datetime` para trabajar con un tipo de dato adecuado.

También se observa que algunas variables numéricas tienen menos registros no nulos que el total de 146.660 filas. Por ejemplo, `lat`, `lon`, `bathrooms`, `surface_total` y `surface_covered` presentan datos faltantes. El tratamiento de estos valores se realizará en su sección correspondiente, sin modificarlos todavía.

Variables como `l1`, `l2`, `l3`, `currency`, `property_type` y `operation_type` representan información categórica, mientras que `title` y `description` corresponden principalmente a texto descriptivo.


## Estadísticos descriptivos

Para comprender mejor las variables numéricas más importantes del dataset se calcularán medidas de tendencia central y dispersión. Se revisarán la media, mediana, moda, desviación estándar, varianza e IQR. Estas medidas ayudan a observar qué tan concentrados o dispersos están los datos y permiten detectar diferencias entre valores típicos y valores extremos.


In [ ]:
variables_analisis = [
    'rooms', 'bedrooms', 'bathrooms',
    'surface_total', 'surface_covered', 'price'
]

estadisticos = pd.DataFrame({
    'media': df[variables_analisis].mean(),
    'mediana': df[variables_analisis].median(),
    'moda': df[variables_analisis].mode().iloc[0],
    'desviacion_estandar': df[variables_analisis].std(),
    'varianza': df[variables_analisis].var(),
    'Q1': df[variables_analisis].quantile(0.25),
    'Q3': df[variables_analisis].quantile(0.75)
})

estadisticos['IQR'] = estadisticos['Q3'] - estadisticos['Q1']
estadisticos.round(2)


### Hallazgos de los estadísticos

Los resultados muestran diferencias importantes entre algunas variables. En `rooms`, la media es aproximadamente **3,08** y la mediana es **3**, por lo que los valores centrales son bastante parecidos. En `bedrooms`, la media es aproximadamente **1,98** y la mediana es **2**.

En cambio, `surface_total` presenta una media cercana a **216,87 m²** y una mediana de **78 m²**. Esta diferencia indica que existen propiedades con superficies muy grandes que elevan la media. Algo parecido ocurre con `surface_covered`, cuya media es aproximadamente **112,82 m²** y su mediana es **68 m²**.

La variable `price` también presenta una diferencia clara entre la media y la mediana. El precio promedio es aproximadamente **USD 241.221**, mientras que la mediana es **USD 166.000**. Además, su desviación estándar es cercana a **USD 318.519**, lo que indica una dispersión alta entre los precios publicados.

Estos resultados sugieren que variables como precio y superficie pueden contener valores extremos o distribuciones asimétricas. En los siguientes apartados se revisarán sus distribuciones y outliers antes de tomar decisiones de limpieza.


## Distribuciones y visualizaciones

Pendiente de desarrollo.


# Fase 3 - Preparación de los Datos

## Missing values

Pendiente de desarrollo.

## Outliers

Pendiente de desarrollo.

## Normalización y estandarización

Pendiente de desarrollo.

## Dataset final

Pendiente de desarrollo.

## Conclusiones

Pendiente de desarrollo.
